# 06D OpenMM Solvated System

This notebook is the explicit-solvent OpenMM production-preparation path. It is separate from 06A because solvated OpenMM needs periodic boxes, PME, water and ion setup, checkpoints, and restart handling.

It writes to `examples/output/md_tests/<SYSTEM>/openmm/solvated_polymer/` and does not submit HPC jobs.


## Workflow Scope

- Engine: OpenMM
- System: explicit solvent polymer
- Solvation API: `openmm.app.Modeller.addSolvent`
- Water/ions: TIP3P-style water, NaCl via `positiveIon` and `negativeIon`
- Nonbonded method: PME
- Platform: CPU locally by default, CUDA when requested
- Outputs: checkpoints, `state.xml`, logs, trajectories, final PDB

Important force-field note: `Modeller.addSolvent` and `ForceField.createSystem` need an OpenMM force field/template setup that can parameterize the PHA polymer and the solvent. Keep `RUN_OPENMM_SOLVATED = False` until that force-field setup is confirmed.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
from iphasimulator.naming import oligomer_name

system_name = oligomer_name("3HB", 4)
md_root = output_root / "md_tests" / system_name
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

openmm_root = md_root / "openmm"
openmm_solvated_dir = openmm_root / "solvated_polymer"
openmm_solvated_dir.mkdir(parents=True, exist_ok=True)

polymer_pdb_path = output_root / "polymer_structures" / f"{system_name}.pdb"

{
    "polymer_pdb": polymer_pdb_path,
    "polymer_pdb_exists": polymer_pdb_path.exists(),
    "openmm_solvated_dir": openmm_solvated_dir,
}

## Solvation Settings

Use small settings for a smoke test. Production settings belong in a config file or script, then execution belongs in notebook 07.


In [ ]:
solvation_settings = {
    "padding_nm": 1.2,
    "ionic_strength_molar": 0.15,
    "temperature_kelvin": 300.0,
    "pressure_bar": 1.0,
    "platform_name": None,  # set to "CUDA" after the CPU path is validated
    "platform_precision": "mixed",
    "minimization_max_iterations": 200,
    "nvt_steps": 100,
    "npt_steps": 100,
    "production_steps": 100,
    "report_interval": 10,
}

solvation_settings

## Build a Solvated OpenMM Model

This is a template for the OpenMM-native solvation route. It stays disabled until the force-field files can parameterize both the PHA polymer and TIP3P water.


In [ ]:
RUN_OPENMM_SOLVATION_SETUP = False

if RUN_OPENMM_SOLVATION_SETUP:
    from openmm import app, unit

    pdb = app.PDBFile(str(polymer_pdb_path))
    forcefield = app.ForceField(
        "amber14-all.xml",
        "amber14/tip3p.xml",
        # Add the polymer template generator or XML force-field file here.
    )
    modeller = app.Modeller(pdb.topology, pdb.positions)
    modeller.addSolvent(
        forcefield,
        model="tip3p",
        padding=solvation_settings["padding_nm"] * unit.nanometer,
        ionicStrength=solvation_settings["ionic_strength_molar"] * unit.molar,
        positiveIon="Na+",
        negativeIon="Cl-",
        neutralize=True,
    )

    solvated_pdb_path = openmm_solvated_dir / "step5_input.pdb"
    with solvated_pdb_path.open("w") as handle:
        app.PDBFile.writeFile(modeller.topology, modeller.positions, handle)
    print(f"Wrote {solvated_pdb_path}")
else:
    print("Set RUN_OPENMM_SOLVATION_SETUP = True after the polymer OpenMM force-field templates are available.")

## Run Solvated OpenMM Dynamics

This template shows the PME/CUDA/checkpoint structure. Keep execution disabled until the solvated model and force-field setup are validated.


In [ ]:
RUN_OPENMM_SOLVATED = False

if RUN_OPENMM_SOLVATED:
    import openmm as mm
    from openmm import app, unit, XmlSerializer

    solvated_pdb_path = openmm_solvated_dir / "step5_input.pdb"
    pdb = app.PDBFile(str(solvated_pdb_path))
    forcefield = app.ForceField("amber14-all.xml", "amber14/tip3p.xml")
    system = forcefield.createSystem(
        pdb.topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0 * unit.nanometer,
        constraints=app.HBonds,
    )
    system.addForce(mm.MonteCarloBarostat(
        solvation_settings["pressure_bar"] * unit.bar,
        solvation_settings["temperature_kelvin"] * unit.kelvin,
    ))
    integrator = mm.LangevinMiddleIntegrator(
        solvation_settings["temperature_kelvin"] * unit.kelvin,
        1.0 / unit.picosecond,
        2.0 * unit.femtoseconds,
    )
    platform = None
    if solvation_settings["platform_name"]:
        platform = mm.Platform.getPlatformByName(solvation_settings["platform_name"])
    simulation = app.Simulation(pdb.topology, system, integrator, platform) if platform else app.Simulation(pdb.topology, system, integrator)
    simulation.context.setPositions(pdb.positions)
    simulation.minimizeEnergy(maxIterations=solvation_settings["minimization_max_iterations"])
    simulation.saveCheckpoint(str(openmm_solvated_dir / "checkpoint.chk"))
    state = simulation.context.getState(getPositions=True, getVelocities=True, getEnergy=True)
    (openmm_solvated_dir / "state.xml").write_text(XmlSerializer.serialize(state))
else:
    print("Set RUN_OPENMM_SOLVATED = True only after setup validation.")